# Matching Experiments

Questo notebook utilizza il prototipo di matching cane–famiglia per confrontare più profili adottanti.

I profili famiglia sono caricati da un file separato (`family_profiles.json`).  
In questo modo è possibile scegliere una famiglia diversa senza modificare la logica del sistema.

Il notebook calcola:

- compatibilità strutturata;
- similarità semantica tramite BERT;
- score finale;
- classifica dei cani consigliati;
- spiegazione automatica della raccomandazione.


## 1. Import delle librerie

In [3]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

## 2. Caricamento dei dati

Vengono caricati:

- il dataset pulito dei cani;
- gli embedding BERT delle descrizioni dei cani;
- il file esterno con i profili famiglia.


In [4]:
dogs = pd.read_csv("../data/processed/dogs_clean.csv")
bert_embeddings = np.load("../data/processed/bert_embeddings.npy")

profiles_path = Path("../data/family_profiles.json")

with open(profiles_path, "r", encoding="utf-8") as f:
    family_profiles = json.load(f)

print("Dataset cani:", dogs.shape)
print("Embedding BERT:", bert_embeddings.shape)
print("Profili famiglia disponibili:", list(family_profiles.keys()))

Dataset cani: (8132, 30)
Embedding BERT: (8132, 768)
Profili famiglia disponibili: ['family_children_apartment', 'single_active_person', 'senior_applicant', 'experienced_countryside_family', 'first_time_adopter']


## 3. Scelta del profilo famiglia

Modifica solo il valore di `selected_profile_id` scegliendo uno degli ID disponibili.

Esempi:

- `family_children_apartment`
- `single_active_person`
- `senior_applicant`
- `experienced_countryside_family`
- `first_time_adopter`


In [5]:
selected_profile_id = "family_children_apartment"

family_profile = family_profiles[selected_profile_id]

family_profile

{'profile_name': 'Famiglia con bambini in appartamento',
 'applicant_age': '31-60',
 'children': True,
 'dog_experience': 'some',
 'housing': 'apartment',
 'garden': False,
 'preferred_gender': 'indifferent',
 'preferred_age': 'young',
 'preferred_size': 'small',
 'preferred_fur': 'short',
 'daily_time': '2-4',
 'activity_level': 'moderate',
 'ideal_dog_description': 'We are looking for a small, affectionate and friendly dog suitable for a family with children. The dog should be calm at home, sociable, easy to manage and suitable for apartment life.'}

## 4. Regole di compatibilità strutturata

La compatibilità strutturata confronta preferenze della famiglia e caratteristiche del cane.

Il sistema distingue tra:

- **vincoli hard**, che possono portare a esclusione o score pari a 0;
- **vincoli soft**, che modificano gradualmente il punteggio.

Regole principali:

- disponibilità inferiore a 1 ora al giorno: adozione non consigliata;
- richiedente over 60: i cuccioli vengono fortemente penalizzati;
- cane extra large senza giardino: esclusione;
- nessuna esperienza con cani extra large: esclusione;
- nessuna esperienza con cani large: forte penalizzazione;
- cane large senza giardino: forte penalizzazione.


In [6]:
def match_preference(preference, dog_value):
    if preference == "indifferent":
        return 1.0
    return 1.0 if preference == dog_value else 0.0


def size_similarity(preferred_size, dog_size):
    if preferred_size == "indifferent":
        return 1.0

    size_order = {
        "small": 0,
        "medium": 1,
        "large": 2,
        "extra_large": 3
    }

    if preferred_size not in size_order or dog_size not in size_order:
        return 0.0

    distance = abs(size_order[preferred_size] - size_order[dog_size])

    if distance == 0:
        return 1.0
    elif distance == 1:
        return 0.5
    else:
        return 0.0


def compute_structured_score(dog, family):
    # Hard constraint: less than 1 hour per day is not compatible with adoption.
    if family["daily_time"] == "<1":
        return 0.0

    # Hard constraint: extra large dogs require a garden.
    if dog["maturity_size_label"] == "extra_large" and not family["garden"]:
        return 0.0

    # Hard constraint: no dog experience and extra large dog.
    if family["dog_experience"] == "none" and dog["maturity_size_label"] == "extra_large":
        return 0.0

    scores = []

    # Direct preferences
    scores.append(match_preference(family["preferred_gender"], dog["gender_label"]))
    scores.append(match_preference(family["preferred_age"], dog["age_group"]))
    scores.append(size_similarity(family["preferred_size"], dog["maturity_size_label"]))
    scores.append(match_preference(family["preferred_fur"], dog["fur_length_label"]))

    # Children rule
    if family["children"]:
        scores.append(1.0 if dog["age_group"] in ["puppy", "young"] else 0.5)

    # Garden and size rule
    if dog["maturity_size_label"] == "large":
        scores.append(1.0 if family["garden"] else 0.2)
    elif dog["maturity_size_label"] == "extra_large":
        scores.append(1.0)
    else:
        scores.append(1.0)

    # Time availability rule
    if family["daily_time"] == "1-2":
        scores.append(0.6 if dog["age_group"] == "puppy" else 0.8)
    else:
        scores.append(1.0)

    # Experience rule
    if family["dog_experience"] == "none":
        if dog["maturity_size_label"] == "large":
            scores.append(0.2)
        elif dog["age_group"] == "puppy":
            scores.append(0.5)
        else:
            scores.append(1.0)
    else:
        scores.append(1.0)

    # Older applicant rule
    if family["applicant_age"] == "over_60":
        scores.append(1.0 if dog["age_group"] in ["adult", "senior"] else 0.4)

    return np.mean(scores)

## 5. Calcolo dello score strutturato

In [7]:
dogs_experiment = dogs.copy()

dogs_experiment["structured_score"] = dogs_experiment.apply(
    lambda row: compute_structured_score(row, family_profile),
    axis=1
)

dogs_experiment[[
    "Name",
    "age_group",
    "gender_label",
    "maturity_size_label",
    "fur_length_label",
    "structured_score"
]].head()

,Name,age_group,gender_label,maturity_size_label,fur_length_label,structured_score
0,Brisco,puppy,male,medium,medium,0.6875
1,Miko,puppy,female,medium,short,0.8125
2,Hunter,puppy,male,medium,short,0.8125
3,Siu Pak & Her 6 Puppies,puppy,female,medium,short,0.8125
4,Bear,puppy,male,medium,short,0.8125


## 6. Similarità semantica tramite BERT

La descrizione del cane ideale della famiglia viene trasformata in embedding BERT.  
Poi viene confrontata con gli embedding delle descrizioni dei cani tramite cosine similarity.


In [8]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name)
bert_model.eval()

print("BERT model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT model loaded.


In [9]:
def get_bert_embedding(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = bert_model(**inputs)

    embedding = outputs.last_hidden_state.mean(dim=1)
    return embedding.squeeze().numpy()


family_embedding = get_bert_embedding(family_profile["ideal_dog_description"])

semantic_scores = cosine_similarity(
    family_embedding.reshape(1, -1),
    bert_embeddings
).flatten()

dogs_experiment["bert_similarity"] = (semantic_scores + 1) / 2

dogs_experiment[["Name", "bert_similarity"]].head()

,Name,bert_similarity
0,Brisco,0.907398
1,Miko,0.876957
2,Hunter,0.925550
3,Siu Pak & Her 6 Puppies,0.822069
4,Bear,0.827598


## 7. Score finale

Lo score finale combina:

- 70% compatibilità strutturata;
- 30% similarità semantica BERT.


In [10]:
dogs_experiment["final_score"] = (
    0.7 * dogs_experiment["structured_score"] +
    0.3 * dogs_experiment["bert_similarity"]
)

dogs_experiment[["Name", "structured_score", "bert_similarity", "final_score"]].head()

,Name,structured_score,bert_similarity,final_score
0,Brisco,0.6875,0.907398,0.753470
1,Miko,0.8125,0.876957,0.831837
2,Hunter,0.8125,0.925550,0.846415
3,Siu Pak & Her 6 Puppies,0.8125,0.822069,0.815371
4,Bear,0.8125,0.827598,0.817029


## 8. Classifica finale

In [11]:
ranking = dogs_experiment.sort_values(
    by="final_score",
    ascending=False
)[[
    "Name",
    "Age",
    "age_group",
    "gender_label",
    "maturity_size_label",
    "fur_length_label",
    "PhotoAmt",
    "structured_score",
    "bert_similarity",
    "final_score",
    "Description"
]].head(10)

ranking

,Name,Age,age_group,gender_label,maturity_size_label,fur_length_label,PhotoAmt,structured_score,bert_similarity,final_score,Description
7084,Lucas,12,young,male,small,short,6.0,1.0,0.940067,0.982020,Lucas is a sturdy little dog. Very well behave...
1103,Rascal,12,young,male,small,short,1.0,1.0,0.939743,0.981923,"This cute dog is well behaved, friendly and ta..."
2281,Zorro Samdani,12,young,male,small,short,4.0,1.0,0.937755,0.981327,"If you want to adopt zorro, you must collect h..."
7630,Zorro,9,young,male,small,short,4.0,1.0,0.937143,0.981143,Zorro is a loving and loyal dog adopted from S...
6640,Mango,9,young,female,small,short,1.0,1.0,0.930017,0.979005,My wife got pregnant and therefore we are unab...
1695,Chihuahua,12,young,female,small,short,2.0,1.0,0.927057,0.978117,Found this abandoned Chihuahua nearby KFC. I p...
7486,Camilo 2,12,young,male,small,short,1.0,1.0,0.922956,0.976887,"Very lively dog, as expected from a JR! We are..."
1986,NaN,12,young,female,small,short,2.0,1.0,0.921230,0.976369,"I'm not sure of the breed, but this dog has be..."
454,Doggie,12,young,male,small,short,1.0,1.0,0.918423,0.975527,Hi everyone..I rescued this stray about 2 mont...
7367,Female Mini Pin,12,young,female,small,short,1.0,1.0,0.917905,0.975372,I need to re-home my one year old female Mini ...


## 9. Spiegazione automatica della raccomandazione

In [12]:
def explain_recommendation(dog, family, all_dogs):
    reasons = []

    if family["preferred_age"] == "indifferent" or dog["age_group"] == family["preferred_age"]:
        reasons.append("fascia d'età compatibile")

    if family["preferred_size"] == "indifferent" or dog["maturity_size_label"] == family["preferred_size"]:
        reasons.append("taglia compatibile")

    if family["preferred_gender"] == "indifferent" or dog["gender_label"] == family["preferred_gender"]:
        reasons.append("sesso compatibile")

    if family["preferred_fur"] == "indifferent" or dog["fur_length_label"] == family["preferred_fur"]:
        reasons.append("lunghezza del pelo compatibile")

    if dog["bert_similarity"] > all_dogs["bert_similarity"].quantile(0.75):
        reasons.append("descrizione semanticamente vicina al cane ideale")

    if dog["structured_score"] == 1.0:
        reasons.append("profilo strutturale altamente compatibile")

    if len(reasons) == 0:
        return "Compatibilità basata sul punteggio complessivo."

    return ", ".join(reasons)


ranking_with_explanations = ranking.copy()

ranking_with_explanations["explanation"] = ranking_with_explanations.apply(
    lambda row: explain_recommendation(row, family_profile, dogs_experiment),
    axis=1
)

ranking_with_explanations[["Name", "final_score", "explanation"]]

,Name,final_score,explanation
7084,Lucas,0.982020,"fascia d'età compatibile, taglia compatibile, ..."
1103,Rascal,0.981923,"fascia d'età compatibile, taglia compatibile, ..."
2281,Zorro Samdani,0.981327,"fascia d'età compatibile, taglia compatibile, ..."
7630,Zorro,0.981143,"fascia d'età compatibile, taglia compatibile, ..."
6640,Mango,0.979005,"fascia d'età compatibile, taglia compatibile, ..."
1695,Chihuahua,0.978117,"fascia d'età compatibile, taglia compatibile, ..."
7486,Camilo 2,0.976887,"fascia d'età compatibile, taglia compatibile, ..."
1986,NaN,0.976369,"fascia d'età compatibile, taglia compatibile, ..."
454,Doggie,0.975527,"fascia d'età compatibile, taglia compatibile, ..."
7367,Female Mini Pin,0.975372,"fascia d'età compatibile, taglia compatibile, ..."


## 10. Confronto rapido tra profili famiglia

Questa cella calcola il miglior cane consigliato per ogni profilo famiglia disponibile.  
Serve per verificare che profili diversi producano ranking diversi.


In [13]:
summary_rows = []

for profile_id, profile in family_profiles.items():
    temp = dogs.copy()

    temp["structured_score"] = temp.apply(
        lambda row: compute_structured_score(row, profile),
        axis=1
    )

    profile_embedding = get_bert_embedding(profile["ideal_dog_description"])

    profile_semantic_scores = cosine_similarity(
        profile_embedding.reshape(1, -1),
        bert_embeddings
    ).flatten()

    temp["bert_similarity"] = (profile_semantic_scores + 1) / 2

    temp["final_score"] = (
        0.7 * temp["structured_score"] +
        0.3 * temp["bert_similarity"]
    )

    best = temp.sort_values("final_score", ascending=False).iloc[0]

    summary_rows.append({
        "profile_id": profile_id,
        "profile_name": profile["profile_name"],
        "best_dog": best["Name"],
        "best_dog_age_group": best["age_group"],
        "best_dog_size": best["maturity_size_label"],
        "final_score": best["final_score"]
    })

summary = pd.DataFrame(summary_rows)
summary

,profile_id,profile_name,best_dog,best_dog_age_group,best_dog_size,final_score
0,family_children_apartment,Famiglia con bambini in appartamento,Lucas,young,small,0.982020
1,single_active_person,Persona singola attiva con giardino,Whis-ky,young,medium,0.983357
2,senior_applicant,Richiedente over 60,Sara,adult,small,0.964915
3,experienced_countryside_family,Famiglia esperta in casa di campagna,Cookie,puppy,large,0.986828
4,first_time_adopter,Prima adozione senza esperienza,NaN,adult,small,0.960118


## 11. Conclusioni

Il notebook permette di testare il sistema di matching su più profili famiglia.

Separare i profili in un file esterno rende il sistema più flessibile: è possibile aggiungere nuovi casi di studio senza modificare il codice del recommendation system.

Il sistema produce una classifica personalizzata usando sia regole interpretabili sia similarità semantica basata su BERT.
